In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [2]:
df=pd.read_csv("houses_flats_final.csv")

In [3]:
df.head()

,bedrooms,baths,price,area_sqft,kitchens,store_rooms,gym,furnishing_score,agePossession,property_type,has_servant_room,luxury_category,floor_category
0,4.0,4.0,4.55,3264.0,1.0,1.0,0,37,New Property,flat,1,Very High,Mid Floor
1,4.0,5.0,4.50,3318.4,1.0,0.0,0,31,New Property,flat,1,Very High,Mid Floor
2,3.0,4.0,3.55,2720.0,1.0,1.0,1,49,New Property,flat,1,Very High,Mid Floor
3,4.0,4.0,4.45,2067.2,2.0,1.0,0,18,New Property,flat,1,Very High,Mid Floor
4,4.0,5.0,4.65,3318.4,1.0,1.0,0,18,New Property,flat,1,Very High,Mid Floor


In [4]:
df.shape

(23081, 13)

In [5]:
df['furnishing_score'].value_counts()

furnishing_score
25    9272
15    3470
0     1738
49    1445
24     983
14     653
39     518
5      488
7      462
6      406
8      402
12     370
3      341
13     301
11     298
10     172
23     148
2      146
18     144
16     144
22     142
9      134
1      130
20     109
4       97
47      88
17      74
37      68
21      55
33      42
19      37
30      34
31      31
28      28
40      19
34      19
32      16
43      13
41      12
26      11
38       8
42       6
27       5
35       1
29       1
Name: count, dtype: int64

In [6]:
df['furnishing_type'] = pd.cut(
    df['furnishing_score'],
    bins=[-1, 0, 10, 25, 100],
    labels=[
        'Unfurnished',
        'Semi-Furnished',
        'Furnished',
        'Luxury Furnished'
    ]
)

In [7]:
df=df.drop(columns='furnishing_score')

In [8]:
df.head()

,bedrooms,baths,price,area_sqft,kitchens,store_rooms,gym,agePossession,property_type,has_servant_room,luxury_category,floor_category,furnishing_type
0,4.0,4.0,4.55,3264.0,1.0,1.0,0,New Property,flat,1,Very High,Mid Floor,Luxury Furnished
1,4.0,5.0,4.50,3318.4,1.0,0.0,0,New Property,flat,1,Very High,Mid Floor,Luxury Furnished
2,3.0,4.0,3.55,2720.0,1.0,1.0,1,New Property,flat,1,Very High,Mid Floor,Luxury Furnished
3,4.0,4.0,4.45,2067.2,2.0,1.0,0,New Property,flat,1,Very High,Mid Floor,Furnished
4,4.0,5.0,4.65,3318.4,1.0,1.0,0,New Property,flat,1,Very High,Mid Floor,Furnished


In [9]:
# Rename columns
df.rename(columns={
    'gym': 'is_gym',
    'has_servant_room': 'is_servant_room'
}, inplace=True)

df['is_gym'] = df['is_gym'].map({1: 'Yes', 0: 'No'})
df['is_servant_room'] = df['is_servant_room'].map({1: 'Yes', 0: 'No'})

In [10]:
df.columns

Index(['bedrooms', 'baths', 'price', 'area_sqft', 'kitchens', 'store_rooms',
       'is_gym', 'agePossession', 'property_type', 'is_servant_room',
       'luxury_category', 'floor_category', 'furnishing_type'],
      dtype='object')

In [11]:
# Split features and target
X = df.drop(columns=['price'])
y = df['price']

In [12]:
# Apply log1p transformation
y_transformed = np.log1p(y)

In [13]:
X.columns

Index(['bedrooms', 'baths', 'area_sqft', 'kitchens', 'store_rooms', 'is_gym',
       'agePossession', 'property_type', 'is_servant_room', 'luxury_category',
       'floor_category', 'furnishing_type'],
      dtype='object')

In [14]:
df.columns

Index(['bedrooms', 'baths', 'price', 'area_sqft', 'kitchens', 'store_rooms',
       'is_gym', 'agePossession', 'property_type', 'is_servant_room',
       'luxury_category', 'floor_category', 'furnishing_type'],
      dtype='object')

In [15]:
df.duplicated().sum()

np.int64(4712)

In [16]:
X.dtypes

bedrooms            float64
baths               float64
area_sqft           float64
kitchens            float64
store_rooms         float64
is_gym               object
agePossession        object
property_type        object
is_servant_room      object
luxury_category      object
floor_category       object
furnishing_type    category
dtype: object

In [17]:
X['floor_category'] = X['floor_category'].fillna('Low Floor')

### ordinal encoding

In [18]:
num_cols = [
    'bedrooms',
    'baths',
    'area_sqft',
    'kitchens',
    'store_rooms'
]

ordinal_cols = [
    'agePossession',
    'furnishing_type',
    'luxury_category',
    'floor_category',
    'is_gym',
    'is_servant_room',
    'property_type'
]



In [19]:
X.isnull().sum()

bedrooms           0
baths              0
area_sqft          0
kitchens           0
store_rooms        0
is_gym             0
agePossession      0
property_type      0
is_servant_room    0
luxury_category    0
floor_category     0
furnishing_type    0
dtype: int64

In [20]:
X['floor_category'].value_counts()

floor_category
Low Floor     18618
Mid Floor      4329
High Floor      134
Name: count, dtype: int64

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OrdinalEncoder(), ordinal_cols),
    ],
    remainder='passthrough'
)

In [22]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [23]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

scores.mean(), scores.std()

(np.float64(0.8372546879314742), np.float64(0.0065823185927148305))

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)

In [25]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedrooms', 'baths',
                                                   'area_sqft', 'kitchens',
                                                   'store_rooms']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category', 'is_gym',
                                                   'is_servant_room',
                                                   'property_type'])])),
                ('regressor', LinearRegression())])

In [26]:
y_pred = pipeline.predict(X_test)

In [27]:
y_pred = np.expm1(y_pred)

In [28]:
mean_absolute_error(np.expm1(y_test), y_pred)

1.711593968831895

In [29]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_transformed, test_size=0.2, random_state=42
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test), y_pred))

    return output

In [32]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

model_dict = {
    'linear_reg': LinearRegression(),
    'svr': SVR(),
    'ridge': Ridge(),
    'lasso': Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest': RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost': XGBRegressor()
}

In [ ]:
model_output = []
for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [ ]:
model_output